# STEP 1A — Open Discovery & Vocabulary Expansion

This notebook is deliberately **not proposition-led**.

Its purpose is to test whether the current Step-1 literature map is still missing a neighbouring scientific literature.

It searches the **causal architecture** broadly:

> evolving human state → selected/available information → learning → future state → future information

without assuming that the right vocabulary is:
- distortion;
- recommender feedback loop;
- yoking;
- entropy;
- recovery;
- fMRI;
- or any current Study 1–4 design.

## What `Run all` does

1. Mounts the same Google Drive evidence-corpus folder.
2. Adds a dedicated `OPEN_DISCOVERY` query layer.
3. Automatically runs each open-discovery query in:
   - OpenAlex;
   - PubMed.
4. Saves every executed search.
5. Saves every retrieval.
6. Conservatively deduplicates candidate publications by DOI or exact normalized title.
7. Stores abstracts and database subject/topic metadata separately from the verified evidence corpus.
8. Produces:
   - recurring vocabulary;
   - database subject terms/topics;
   - papers bridging multiple searches/literatures;
   - query-overlap diagnostics.
9. Exports `Step1A_Open_Discovery_Report.xlsx`.

It does **not**:
- screen papers in or out;
- promote candidates into verified publications;
- assign scientific truth;
- decide novelty;
- decide Step-1 closure.

After this report is produced, review the new vocabulary and literature clusters before running the full proposition-directed Step-1 harvest.

## 1. Install

In [ ]:
!pip -q install pandas numpy requests openpyxl biopython rapidfuzz scikit-learn

## 2. Mount Drive and configure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import requests, re, json, os, time
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer

# CHANGE THIS before running.
EMAIL = "your_email@example.com"

ROOT = Path('/content/drive/MyDrive/Step1_Evidence_Corpus')
ROOT.mkdir(parents=True, exist_ok=True)

# Safe defaults for an open-discovery pass.
MAX_PUBMED_PER_QUERY = 100
MAX_OPENALEX_PER_QUERY = 100

# If False, an already-executed query/database combination is skipped.
FORCE_RERUN = False

print("Corpus folder:", ROOT)

## 3. Open-discovery search plan

These searches intentionally extend beyond the current proposal vocabulary.

They include:
- generic state-contingent information environments;
- adaptive education/intelligent tutoring;
- performative prediction/endogenous distribution shift;
- selective exposure and information avoidance;
- adaptive assessment;
- agency/self-generation;
- perceived personalisation/algorithm awareness;
- path-dependent/sequential observation environments.

The list is a **discovery seed**, not an exhaustive literature map.

In [ ]:
OPEN_DISCOVERY_PLAN = [
    {
        "query_plan_id":"ODQ001",
        "domain":"open discovery",
        "planned_query":"adaptive information environment human learning behavior future information",
        "rationale":"Search causal structure without recommender-specific vocabulary."
    },
    {
        "query_plan_id":"ODQ002",
        "domain":"open discovery",
        "planned_query":"user contingent information exposure learning behavior",
        "rationale":"Search information exposure contingent on the same user's behaviour."
    },
    {
        "query_plan_id":"ODQ003",
        "domain":"open discovery",
        "planned_query":"endogenous information acquisition learning beliefs behavior",
        "rationale":"Search economics/decision/cognitive vocabulary for self-selected evidence."
    },
    {
        "query_plan_id":"ODQ004",
        "domain":"adaptive education",
        "planned_query":"adaptive learning system learner state content sequencing experiment",
        "rationale":"Learner-state-driven selection of future learning material."
    },
    {
        "query_plan_id":"ODQ005",
        "domain":"adaptive education",
        "planned_query":"intelligent tutoring learner model adaptive content selection human learning",
        "rationale":"Intelligent tutoring and learner-model literature."
    },
    {
        "query_plan_id":"ODQ006",
        "domain":"adaptive education",
        "planned_query":"personalized curriculum sequencing adaptive practice scheduling learning",
        "rationale":"Adaptive curriculum/practice sequencing."
    },
    {
        "query_plan_id":"ODQ007",
        "domain":"performative prediction",
        "planned_query":"performative prediction feedback distribution shift human behavior",
        "rationale":"Systems whose outputs change their own future data distribution."
    },
    {
        "query_plan_id":"ODQ008",
        "domain":"performative prediction",
        "planned_query":"endogenous data feedback prediction induced distribution shift decision making",
        "rationale":"Alternative vocabulary for feedback-induced distribution change."
    },
    {
        "query_plan_id":"ODQ009",
        "domain":"selective exposure",
        "planned_query":"selective exposure information search belief learning confirmation bias",
        "rationale":"Self-confirming/confirmatory information acquisition."
    },
    {
        "query_plan_id":"ODQ010",
        "domain":"selective exposure",
        "planned_query":"information avoidance information seeking belief updating learning",
        "rationale":"Avoidance as an alternative information-acquisition process."
    },
    {
        "query_plan_id":"ODQ011",
        "domain":"agency and self generation",
        "planned_query":"self generation choice agency active passive learning memory experiment",
        "rationale":"Agency/self-generation effects independent of recommendation."
    },
    {
        "query_plan_id":"ODQ012",
        "domain":"agency and self generation",
        "planned_query":"volitional learning active acquisition passive exposure memory category",
        "rationale":"Volitional versus passive acquisition."
    },
    {
        "query_plan_id":"ODQ013",
        "domain":"algorithm awareness",
        "planned_query":"perceived personalization algorithm awareness behavior learning",
        "rationale":"Psychological effects of knowing/inferencing personalisation."
    },
    {
        "query_plan_id":"ODQ014",
        "domain":"algorithm awareness",
        "planned_query":"perceived algorithm responsiveness personalization cues user behavior",
        "rationale":"Perceived contingency as a rival pathway."
    },
    {
        "query_plan_id":"ODQ015",
        "domain":"adaptive assessment",
        "planned_query":"computerized adaptive testing item selection learning testing effect",
        "rationale":"State-contingent item selection intersecting with measurement reactivity."
    },
    {
        "query_plan_id":"ODQ016",
        "domain":"adaptive assessment",
        "planned_query":"adaptive assessment learner state item sequencing measurement reactivity",
        "rationale":"Adaptive assessment and learning interaction."
    },
    {
        "query_plan_id":"ODQ017",
        "domain":"closed-loop human systems",
        "planned_query":"closed loop adaptive system human behavior future observations learning",
        "rationale":"Broad cross-disciplinary closed-loop search."
    },
    {
        "query_plan_id":"ODQ018",
        "domain":"closed-loop human systems",
        "planned_query":"human behavior changes future information environment feedback learning",
        "rationale":"Search without algorithm-specific terminology."
    },
    {
        "query_plan_id":"ODQ019",
        "domain":"path dependence",
        "planned_query":"path dependence sequential information exposure human learning",
        "rationale":"Early-history/stochastic-path literature."
    },
    {
        "query_plan_id":"ODQ020",
        "domain":"path dependence",
        "planned_query":"history dependent information environment learning decision making",
        "rationale":"History-dependent observation/exposure environments."
    }
]

open_plan_df = pd.DataFrame(OPEN_DISCOVERY_PLAN)
print("Open-discovery queries:", len(open_plan_df))
display(open_plan_df)

## 4. Load/create the persistent corpus tables

In [ ]:
QUERY_PLAN_COLUMNS = [
    "query_plan_id","domain","search_purpose","planned_query",
    "preferred_sources","priority","rationale","status","notes"
]
SEARCH_COLUMNS = [
    "search_id","query_plan_id","domain","database","date_searched",
    "query","filters","search_purpose","direct_or_adjacent",
    "total_results_reported","results_retrieved","results_screened",
    "backward_checked","forward_checked","notes"
]
RETRIEVAL_COLUMNS = [
    "retrieval_id","search_id","candidate_id","retrieval_rank",
    "retrieval_method","database_record_id","notes"
]
CANDIDATE_COLUMNS = [
    "candidate_id","title","authors","year","doi","url",
    "screening_decision","exclusion_reason","duplicate_of",
    "linked_publication_id","notes"
]
PROPOSITION_COLUMNS = [
    "proposition_tag","plain_language_question","domain","active","notes"
]
DISCOVERY_RECORD_COLUMNS = [
    "discovery_record_id","candidate_id","database","database_record_id",
    "abstract","subject_terms","source_topics","publication_type",
    "cited_by_count","retrieved_date","notes"
]

SCHEMAS = {
    "query_plan": QUERY_PLAN_COLUMNS,
    "searches": SEARCH_COLUMNS,
    "retrievals": RETRIEVAL_COLUMNS,
    "candidates": CANDIDATE_COLUMNS,
    "propositions": PROPOSITION_COLUMNS,
    "discovery_records": DISCOVERY_RECORD_COLUMNS
}

def path_for(name):
    return ROOT / f"{name}.csv"

def load_or_create(name, columns):
    p = path_for(name)
    if p.exists():
        df = pd.read_csv(p, dtype=str).fillna("")
        for c in columns:
            if c not in df.columns:
                df[c] = ""
        return df[columns]
    df = pd.DataFrame(columns=columns)
    df.to_csv(p, index=False)
    return df

tables = {name:load_or_create(name,cols) for name,cols in SCHEMAS.items()}

# Add open-discovery query plan without overwriting the existing 121-query plan.
existing_qp = set(tables["query_plan"]["query_plan_id"].astype(str))
new_rows = []
for q in OPEN_DISCOVERY_PLAN:
    if q["query_plan_id"] not in existing_qp:
        new_rows.append({
            "query_plan_id":q["query_plan_id"],
            "domain":q["domain"],
            "search_purpose":"OPEN_DISCOVERY",
            "planned_query":q["planned_query"],
            "preferred_sources":"OpenAlex;PubMed",
            "priority":"HIGH",
            "rationale":q["rationale"],
            "status":"PLANNED",
            "notes":"Step 1A vocabulary-expansion search"
        })

if new_rows:
    tables["query_plan"] = pd.concat(
        [tables["query_plan"],pd.DataFrame(new_rows)],
        ignore_index=True
    )
    tables["query_plan"].to_csv(path_for("query_plan"),index=False)

# Add an explicit emergent/unclassified route if the proposition registry already exists.
if len(tables["propositions"]) == 0:
    tables["propositions"] = pd.DataFrame([{
        "proposition_tag":"PT000",
        "plain_language_question":"EMERGENT / NOT YET CLASSIFIED",
        "domain":"open discovery",
        "active":"yes",
        "notes":"Temporary route for findings that do not fit the current proposition map."
    }])
elif "PT000" not in set(tables["propositions"]["proposition_tag"].astype(str)):
    tables["propositions"] = pd.concat([
        tables["propositions"],
        pd.DataFrame([{
            "proposition_tag":"PT000",
            "plain_language_question":"EMERGENT / NOT YET CLASSIFIED",
            "domain":"open discovery",
            "active":"yes",
            "notes":"Temporary route for findings that do not fit the current proposition map."
        }])
    ],ignore_index=True)

tables["propositions"].to_csv(path_for("propositions"),index=False)

print({name:len(df) for name,df in tables.items()})

## 5. Persistent IDs and normalization

In [ ]:
PREFIX = {
    "searches":"S",
    "retrievals":"RET",
    "candidates":"C",
    "discovery_records":"DR"
}
ID_COL = {
    "searches":"search_id",
    "retrievals":"retrieval_id",
    "candidates":"candidate_id",
    "discovery_records":"discovery_record_id"
}

def save(name):
    tables[name].to_csv(path_for(name),index=False)

def next_id(name):
    prefix = PREFIX[name]
    col = ID_COL[name]
    nums = []
    for v in tables[name][col].astype(str):
        m = re.search(r"(\d+)$",v)
        if m:
            nums.append(int(m.group(1)))
    return f"{prefix}{max(nums,default=0)+1:05d}"

def normalize_doi(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"^https?://(dx\.)?doi\.org/","",x)
    x = re.sub(r"^doi:\s*","",x)
    return x.strip()

def normalize_title(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x or "").lower()).strip()

def safe_text(x):
    if x is None:
        return ""
    return str(x)

## 6. PubMed retrieval with abstracts and MeSH terms

In [ ]:
from Bio import Entrez
Entrez.email = EMAIL

def pubmed_search_records(query, retmax=100):
    h = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=retmax,
        sort="relevance"
    )
    search_res = Entrez.read(h)
    ids = list(search_res.get("IdList",[]))
    total = int(search_res.get("Count",0))
    if not ids:
        return total, pd.DataFrame()

    time.sleep(0.35)
    h = Entrez.efetch(
        db="pubmed",
        id=",".join(ids),
        rettype="xml"
    )
    data = Entrez.read(h)

    rows = []
    for pa in data.get("PubmedArticle",[]):
        try:
            med = pa["MedlineCitation"]
            art = med["Article"]
            pmid = safe_text(med.get("PMID",""))
            title = safe_text(art.get("ArticleTitle",""))

            # Abstract
            abstract_parts = []
            abstract_obj = art.get("Abstract",{})
            for part in abstract_obj.get("AbstractText",[]) if abstract_obj else []:
                label = getattr(part,"attributes",{}).get("Label","")
                txt = safe_text(part)
                abstract_parts.append((label + ": " if label else "") + txt)
            abstract = " ".join(abstract_parts)

            # Authors
            author_names = []
            for a in art.get("AuthorList",[]) or []:
                collective = safe_text(a.get("CollectiveName",""))
                if collective:
                    author_names.append(collective)
                else:
                    last = safe_text(a.get("LastName",""))
                    initials = safe_text(a.get("Initials",""))
                    name = " ".join(x for x in [last,initials] if x)
                    if name:
                        author_names.append(name)

            # DOI and other article IDs
            doi = ""
            article_ids = pa.get("PubmedData",{}).get("ArticleIdList",[])
            for aid in article_ids:
                attrs = getattr(aid,"attributes",{})
                if attrs.get("IdType") == "doi":
                    doi = normalize_doi(aid)
                    break

            # Year
            year = ""
            journal_issue = art.get("Journal",{}).get("JournalIssue",{})
            pubdate = journal_issue.get("PubDate",{}) if journal_issue else {}
            year = safe_text(pubdate.get("Year",""))
            if not year:
                medline_date = safe_text(pubdate.get("MedlineDate",""))
                m = re.search(r"(19|20)\d{2}",medline_date)
                year = m.group(0) if m else ""

            # MeSH
            mesh_terms = []
            for mh in med.get("MeshHeadingList",[]) or []:
                desc = safe_text(mh.get("DescriptorName",""))
                if desc:
                    mesh_terms.append(desc)

            pubtypes = [safe_text(x) for x in art.get("PublicationTypeList",[]) or []]

            rows.append({
                "database":"PubMed",
                "database_record_id":pmid,
                "title":title,
                "authors":"; ".join(author_names),
                "year":year,
                "doi":doi,
                "url":f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else "",
                "abstract":abstract,
                "subject_terms":"; ".join(sorted(set(mesh_terms))),
                "source_topics":"",
                "publication_type":"; ".join(pubtypes),
                "cited_by_count":""
            })
        except Exception as e:
            rows.append({
                "database":"PubMed",
                "database_record_id":"",
                "title":"",
                "authors":"",
                "year":"",
                "doi":"",
                "url":"",
                "abstract":"",
                "subject_terms":"",
                "source_topics":"",
                "publication_type":"",
                "cited_by_count":"",
                "parse_error":str(e)
            })

    return total, pd.DataFrame(rows)

## 7. OpenAlex retrieval with abstracts/topics

In [ ]:
def reconstruct_openalex_abstract(inv):
    if not inv:
        return ""
    max_pos = -1
    for positions in inv.values():
        if positions:
            max_pos = max(max_pos,max(positions))
    if max_pos < 0:
        return ""
    words = [""]*(max_pos+1)
    for word,positions in inv.items():
        for p in positions:
            if 0 <= p < len(words):
                words[p] = word
    return " ".join(w for w in words if w)

def openalex_search_records(query, max_records=100):
    rows = []
    cursor = "*"
    total = None

    while len(rows) < max_records:
        per_page = min(100,max_records-len(rows))
        params = {
            "search":query,
            "per-page":per_page,
            "cursor":cursor,
            "mailto":EMAIL
        }
        r = requests.get("https://api.openalex.org/works",params=params,timeout=45)
        r.raise_for_status()
        data = r.json()
        if total is None:
            total = data.get("meta",{}).get("count",0)

        batch = data.get("results",[])
        if not batch:
            break

        for w in batch:
            authors = []
            for a in w.get("authorships",[]) or []:
                nm = (a.get("author") or {}).get("display_name","")
                if nm:
                    authors.append(nm)

            topics = [
                t.get("display_name","")
                for t in (w.get("topics",[]) or [])[:10]
                if t.get("display_name")
            ]
            keywords = [
                k.get("display_name","")
                for k in (w.get("keywords",[]) or [])[:20]
                if k.get("display_name")
            ]

            rows.append({
                "database":"OpenAlex",
                "database_record_id":safe_text(w.get("id","")),
                "title":safe_text(w.get("display_name","")),
                "authors":"; ".join(authors),
                "year":safe_text(w.get("publication_year","")),
                "doi":normalize_doi(w.get("doi","")),
                "url":safe_text((w.get("primary_location") or {}).get("landing_page_url","")),
                "abstract":reconstruct_openalex_abstract(w.get("abstract_inverted_index")),
                "subject_terms":"; ".join(keywords),
                "source_topics":"; ".join(topics),
                "publication_type":safe_text(w.get("type","")),
                "cited_by_count":safe_text(w.get("cited_by_count",""))
            })
            if len(rows) >= max_records:
                break

        cursor = data.get("meta",{}).get("next_cursor")
        if not cursor:
            break
        time.sleep(0.15)

    return int(total or 0), pd.DataFrame(rows)

## 8. Search/candidate/retrieval storage

In [ ]:
def already_searched(query_plan_id,database):
    if FORCE_RERUN:
        return False
    s = tables["searches"]
    if not len(s):
        return False
    return (
        (s["query_plan_id"].astype(str)==str(query_plan_id)) &
        (s["database"].astype(str).str.lower()==str(database).lower())
    ).any()

def add_search_log(query_plan_id,domain,database,query,total,retrieved,notes=""):
    sid = next_id("searches")
    row = {
        "search_id":sid,
        "query_plan_id":query_plan_id,
        "domain":domain,
        "database":database,
        "date_searched":pd.Timestamp.now().date().isoformat(),
        "query":query,
        "filters":"",
        "search_purpose":"OPEN_DISCOVERY",
        "direct_or_adjacent":"OPEN",
        "total_results_reported":str(total),
        "results_retrieved":str(retrieved),
        "results_screened":"0",
        "backward_checked":"",
        "forward_checked":"",
        "notes":notes
    }
    tables["searches"] = pd.concat([tables["searches"],pd.DataFrame([row])],ignore_index=True)
    save("searches")
    return sid

def find_existing_candidate(title,doi):
    doi = normalize_doi(doi)
    c = tables["candidates"]
    if not len(c):
        return None

    if doi:
        d = c["doi"].astype(str).map(normalize_doi)
        hit = c[d==doi]
        if len(hit):
            return hit.iloc[0]["candidate_id"]

    nt = normalize_title(title)
    if nt:
        titles = c["title"].astype(str).map(normalize_title)
        hit = c[titles==nt]
        if len(hit):
            return hit.iloc[0]["candidate_id"]
    return None

def get_or_create_candidate(row):
    existing = find_existing_candidate(row.get("title",""),row.get("doi",""))
    if existing:
        return existing

    cid = next_id("candidates")
    new = {
        "candidate_id":cid,
        "title":row.get("title",""),
        "authors":row.get("authors",""),
        "year":row.get("year",""),
        "doi":normalize_doi(row.get("doi","")),
        "url":row.get("url",""),
        "screening_decision":"NOT_SCREENED",
        "exclusion_reason":"",
        "duplicate_of":"",
        "linked_publication_id":"",
        "notes":"Automatically harvested during Step 1A open discovery."
    }
    tables["candidates"] = pd.concat([tables["candidates"],pd.DataFrame([new])],ignore_index=True)
    save("candidates")
    return cid

def add_retrieval(search_id,candidate_id,row,rank):
    # Do not duplicate an identical retrieval.
    r = tables["retrievals"]
    if len(r):
        same = (
            (r["search_id"].astype(str)==str(search_id)) &
            (r["candidate_id"].astype(str)==str(candidate_id)) &
            (r["database_record_id"].astype(str)==str(row.get("database_record_id","")))
        )
        if same.any():
            return r[same].iloc[0]["retrieval_id"]

    rid = next_id("retrievals")
    new = {
        "retrieval_id":rid,
        "search_id":search_id,
        "candidate_id":candidate_id,
        "retrieval_rank":str(rank),
        "retrieval_method":"open discovery database search",
        "database_record_id":row.get("database_record_id",""),
        "notes":""
    }
    tables["retrievals"] = pd.concat([tables["retrievals"],pd.DataFrame([new])],ignore_index=True)
    save("retrievals")
    return rid

def add_discovery_record(candidate_id,row):
    d = tables["discovery_records"]
    db = str(row.get("database",""))
    recid = str(row.get("database_record_id",""))

    if len(d):
        same = (
            (d["candidate_id"].astype(str)==str(candidate_id)) &
            (d["database"].astype(str)==db) &
            (d["database_record_id"].astype(str)==recid)
        )
        if same.any():
            return d[same].iloc[0]["discovery_record_id"]

    did = next_id("discovery_records")
    new = {
        "discovery_record_id":did,
        "candidate_id":candidate_id,
        "database":db,
        "database_record_id":recid,
        "abstract":row.get("abstract",""),
        "subject_terms":row.get("subject_terms",""),
        "source_topics":row.get("source_topics",""),
        "publication_type":row.get("publication_type",""),
        "cited_by_count":row.get("cited_by_count",""),
        "retrieved_date":pd.Timestamp.now().date().isoformat(),
        "notes":"Discovery metadata only; not a verified evidence extraction."
    }
    tables["discovery_records"] = pd.concat(
        [tables["discovery_records"],pd.DataFrame([new])],
        ignore_index=True
    )
    save("discovery_records")
    return did

## 9. One-click open-discovery driver

This is the cell that makes `Run all` perform the actual Step 1A harvest.

It is restart-safe:
- existing query/database combinations are skipped;
- candidates are reused by DOI or exact normalized title;
- retrieval links are preserved separately.

Set `FORCE_RERUN=True` only if you deliberately want fresh duplicate search executions.

In [ ]:
def run_open_discovery():
    summary = []

    for q in OPEN_DISCOVERY_PLAN:
        qid = q["query_plan_id"]
        domain = q["domain"]
        query = q["planned_query"]

        for database in ["OpenAlex","PubMed"]:
            if already_searched(qid,database):
                print("SKIP",qid,database,"already executed")
                continue

            print("\nRUN",qid,database,":",query)

            try:
                if database == "OpenAlex":
                    total,hits = openalex_search_records(
                        query,
                        max_records=MAX_OPENALEX_PER_QUERY
                    )
                else:
                    total,hits = pubmed_search_records(
                        query,
                        retmax=MAX_PUBMED_PER_QUERY
                    )

                sid = add_search_log(
                    qid,domain,database,query,total,len(hits)
                )

                for rank,(_,row) in enumerate(hits.iterrows(),start=1):
                    cid = get_or_create_candidate(row)
                    add_retrieval(sid,cid,row,rank)
                    add_discovery_record(cid,row)

                summary.append({
                    "query_plan_id":qid,
                    "database":database,
                    "total_results":total,
                    "retrieved":len(hits),
                    "status":"OK"
                })
                print("  total:",total,"retrieved:",len(hits))

            except Exception as e:
                # Log failed execution so the failure is visible.
                sid = add_search_log(
                    qid,domain,database,query,"","",
                    notes="ERROR: "+str(e)
                )
                summary.append({
                    "query_plan_id":qid,
                    "database":database,
                    "total_results":"",
                    "retrieved":0,
                    "status":"ERROR: "+str(e)
                })
                print("  ERROR:",e)

            time.sleep(0.4)

    return pd.DataFrame(summary)

open_discovery_run_summary = run_open_discovery()
display(open_discovery_run_summary)

## 10. Conservative duplicate-candidate report

In [ ]:
def duplicate_candidate_report(title_threshold=96):
    c = tables["candidates"].reset_index(drop=True)
    rows = []
    for i in range(len(c)):
        for j in range(i+1,len(c)):
            di = normalize_doi(c.loc[i,"doi"])
            dj = normalize_doi(c.loc[j,"doi"])
            same_doi = bool(di and dj and di==dj)

            ti = normalize_title(c.loc[i,"title"])
            tj = normalize_title(c.loc[j,"title"])
            sim = fuzz.ratio(ti,tj) if ti and tj else 0

            # Candidate creation already merges exact DOI/title.
            # This report only surfaces likely remaining duplicates for manual review.
            if same_doi or sim >= title_threshold:
                rows.append({
                    "candidate_1":c.loc[i,"candidate_id"],
                    "candidate_2":c.loc[j,"candidate_id"],
                    "same_doi":same_doi,
                    "title_similarity":sim,
                    "title_1":c.loc[i,"title"],
                    "title_2":c.loc[j,"title"]
                })
    return pd.DataFrame(rows)

duplicate_candidates = duplicate_candidate_report()
print("Possible remaining duplicates:",len(duplicate_candidates))

## 11. Build one discovery text record per candidate

This combines database abstracts/subject metadata for vocabulary discovery only.

It does not overwrite the later full-text evidence extraction.

In [ ]:
def candidate_discovery_text():
    c = tables["candidates"][["candidate_id","title","authors","year","doi","url"]].copy()
    d = tables["discovery_records"].copy()

    grouped = []
    for cid,g in d.groupby("candidate_id"):
        abstracts = [x for x in g["abstract"].astype(str) if x.strip()]
        terms = []
        topics = []
        databases = sorted(set(g["database"].astype(str)))
        for x in g["subject_terms"].astype(str):
            terms.extend([t.strip() for t in x.split(";") if t.strip()])
        for x in g["source_topics"].astype(str):
            topics.extend([t.strip() for t in x.split(";") if t.strip()])

        grouped.append({
            "candidate_id":cid,
            "abstract":" ".join(dict.fromkeys(abstracts)),
            "subject_terms":"; ".join(sorted(set(terms))),
            "source_topics":"; ".join(sorted(set(topics))),
            "databases":"; ".join(databases)
        })

    gdf = pd.DataFrame(grouped)
    if not len(gdf):
        c["abstract"]=""
        c["subject_terms"]=""
        c["source_topics"]=""
        c["databases"]=""
        c["discovery_text"]=c["title"].astype(str)
        return c

    out = c.merge(gdf,on="candidate_id",how="left").fillna("")
    out["discovery_text"] = (
        out["title"].astype(str) + ". " +
        out["abstract"].astype(str) + ". " +
        out["subject_terms"].astype(str) + ". " +
        out["source_topics"].astype(str)
    )
    return out

discovery_text_df = candidate_discovery_text()
print("Candidates with discovery text:",len(discovery_text_df))

## 12. Emergent vocabulary report

This is a lexical surfacing tool, not a scientific topic model.

It identifies recurring 1–3 word terms in titles/abstracts/subject metadata and reports:
- document frequency;
- mean TF-IDF weight;
- whether the phrase already occurs in the seeded open-discovery queries;
- example papers.

Terms not present in the seed queries are especially useful for vocabulary expansion, but **frequency does not imply scientific importance**.

In [ ]:
def emergent_vocabulary_report(df, max_terms=300):
    if len(df) < 2:
        return pd.DataFrame()

    texts = df["discovery_text"].fillna("").astype(str).tolist()

    # Avoid only generic English function words. Do not delete scientific anchor words;
    # instead mark whether terms overlap with the seed queries.
    min_df = 2 if len(texts) >= 10 else 1

    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,3),
        min_df=min_df,
        max_df=0.95,
        max_features=8000,
        lowercase=True
    )

    try:
        X = vec.fit_transform(texts)
    except ValueError:
        return pd.DataFrame()

    terms = np.array(vec.get_feature_names_out())
    doc_freq = np.asarray((X>0).sum(axis=0)).ravel()
    mean_score = np.asarray(X.mean(axis=0)).ravel()

    seed_text = " ".join(
        normalize_title(q["planned_query"])
        for q in OPEN_DISCOVERY_PLAN
    )

    order = np.lexsort((-mean_score,-doc_freq))
    order = order[:max_terms]

    rows = []
    for idx in order:
        term = terms[idx]
        col = X[:,idx].toarray().ravel()
        top_idx = np.argsort(-col)[:3]
        examples = [
            df.iloc[i]["title"]
            for i in top_idx
            if col[i] > 0
        ]
        rows.append({
            "term":term,
            "document_frequency":int(doc_freq[idx]),
            "document_prevalence":round(float(doc_freq[idx]/len(df)),4),
            "mean_tfidf":round(float(mean_score[idx]),6),
            "already_in_seed_queries":"yes" if normalize_title(term) in seed_text else "no",
            "example_titles":" || ".join(examples),
            "manual_new_literature_flag":"",
            "manual_followup_query":"",
            "notes":""
        })

    return pd.DataFrame(rows)

emergent_terms_df = emergent_vocabulary_report(discovery_text_df)
print("Emergent terms reported:",len(emergent_terms_df))
display(emergent_terms_df.head(40))

## 13. Database-supplied subject/topic report

In [ ]:
def subject_topic_report(df, max_terms=300):
    counter = {}
    examples = {}

    for _,r in df.iterrows():
        cid = r["candidate_id"]
        title = r["title"]
        vals = []
        for col in ["subject_terms","source_topics"]:
            vals.extend([x.strip() for x in str(r.get(col,"")).split(";") if x.strip()])

        for term in set(vals):
            key = term.strip()
            if not key:
                continue
            counter[key] = counter.get(key,0)+1
            examples.setdefault(key,[])
            if len(examples[key]) < 3:
                examples[key].append(title)

    rows = [
        {
            "database_subject_or_topic":term,
            "candidate_count":count,
            "example_titles":" || ".join(examples.get(term,[])),
            "manual_new_literature_flag":"",
            "manual_followup_query":"",
            "notes":""
        }
        for term,count in counter.items()
    ]
    return pd.DataFrame(rows).sort_values(
        ["candidate_count","database_subject_or_topic"],
        ascending=[False,True]
    ).head(max_terms) if rows else pd.DataFrame()

subject_topics_df = subject_topic_report(discovery_text_df)
print("Database topic/subject terms:",len(subject_topics_df))
display(subject_topics_df.head(40))

## 14. Bridging papers

Papers retrieved by several different open-discovery queries are useful for locating literatures that connect otherwise separate domains.

This does not make them stronger evidence; it only makes them useful vocabulary/citation-chain bridges.

In [ ]:
def bridge_candidate_report():
    ret = tables["retrievals"].copy()
    sea = tables["searches"].copy()
    can = tables["candidates"].copy()

    if not len(ret):
        return pd.DataFrame()

    x = ret.merge(
        sea[["search_id","query_plan_id","domain","database","query"]],
        on="search_id",how="left"
    )

    rows = []
    for cid,g in x.groupby("candidate_id"):
        qids = sorted(set(g["query_plan_id"].astype(str)))
        domains = sorted(set(g["domain"].astype(str)))
        dbs = sorted(set(g["database"].astype(str)))
        if len(qids) < 2:
            continue

        cr = can[can["candidate_id"].astype(str)==str(cid)]
        if not len(cr):
            continue

        rows.append({
            "candidate_id":cid,
            "title":cr.iloc[0]["title"],
            "doi":cr.iloc[0]["doi"],
            "query_count":len(qids),
            "domain_count":len(domains),
            "query_plan_ids":"; ".join(qids),
            "domains":"; ".join(domains),
            "databases":"; ".join(dbs)
        })

    return pd.DataFrame(rows).sort_values(
        ["domain_count","query_count"],
        ascending=[False,False]
    ) if rows else pd.DataFrame()

bridge_candidates_df = bridge_candidate_report()
print("Candidates retrieved by >=2 discovery queries:",len(bridge_candidates_df))
display(bridge_candidates_df.head(40))

## 15. Query-overlap and harvest diagnostics

In [ ]:
def query_diagnostics():
    s = tables["searches"]
    r = tables["retrievals"]
    rows = []

    for q in OPEN_DISCOVERY_PLAN:
        qid = q["query_plan_id"]
        for db in ["OpenAlex","PubMed"]:
            hits = s[
                (s["query_plan_id"].astype(str)==qid) &
                (s["database"].astype(str)==db)
            ]
            if not len(hits):
                rows.append({
                    "query_plan_id":qid,
                    "domain":q["domain"],
                    "database":db,
                    "executed":"no",
                    "total_results":"",
                    "retrieved":"",
                    "unique_candidates":"",
                    "notes":""
                })
                continue

            # Latest execution if FORCE_RERUN created more than one.
            row = hits.iloc[-1]
            sid = row["search_id"]
            rr = r[r["search_id"].astype(str)==str(sid)]
            rows.append({
                "query_plan_id":qid,
                "domain":q["domain"],
                "database":db,
                "executed":"yes",
                "total_results":row["total_results_reported"],
                "retrieved":row["results_retrieved"],
                "unique_candidates":rr["candidate_id"].nunique(),
                "notes":row["notes"]
            })

    return pd.DataFrame(rows)

query_diagnostics_df = query_diagnostics()
display(query_diagnostics_df)

## 16. Open-discovery integrity checks

In [ ]:
def discovery_integrity_checks():
    issues = []

    search_ids = set(tables["searches"]["search_id"].astype(str))
    candidate_ids = set(tables["candidates"]["candidate_id"].astype(str))

    for _,r in tables["retrievals"].iterrows():
        if str(r["search_id"]) not in search_ids:
            issues.append({
                "table":"retrievals",
                "id":r["retrieval_id"],
                "issue":"missing search link"
            })
        if str(r["candidate_id"]) not in candidate_ids:
            issues.append({
                "table":"retrievals",
                "id":r["retrieval_id"],
                "issue":"missing candidate link"
            })

    for _,r in tables["discovery_records"].iterrows():
        if str(r["candidate_id"]) not in candidate_ids:
            issues.append({
                "table":"discovery_records",
                "id":r["discovery_record_id"],
                "issue":"missing candidate link"
            })

    # Detect queries that retrieved fewer results than their reported total.
    # This is not an error; it is an explicit coverage warning.
    od = tables["searches"][
        tables["searches"]["search_purpose"].astype(str)=="OPEN_DISCOVERY"
    ]
    for _,r in od.iterrows():
        try:
            total = int(float(r["total_results_reported"]))
            got = int(float(r["results_retrieved"]))
            if total > got:
                issues.append({
                    "table":"searches",
                    "id":r["search_id"],
                    "issue":f"partial retrieval: {got} of {total} reported results"
                })
        except Exception:
            pass

    return pd.DataFrame(issues)

open_discovery_integrity_df = discovery_integrity_checks()
display(open_discovery_integrity_df.head(50))

## 17. Export the Step 1A report

Upload this workbook back into the chat.

The next scientific step is to inspect:
- emergent vocabulary;
- database topics;
- bridging papers;
- high-yield unfamiliar literatures;
- searches with very large result spaces.

Only after that should the main Step-1 search map be frozen and the full evidence harvest run.

In [ ]:
def export_open_discovery_report():
    out = ROOT / "Step1A_Open_Discovery_Report.xlsx"

    # Only Step-1A search/retrieval rows in report, while candidates can include
    # pre-existing corpus candidates because deduplication may reuse them.
    od_searches = tables["searches"][
        tables["searches"]["search_purpose"].astype(str)=="OPEN_DISCOVERY"
    ].copy()
    od_search_ids = set(od_searches["search_id"].astype(str))
    od_retrievals = tables["retrievals"][
        tables["retrievals"]["search_id"].astype(str).isin(od_search_ids)
    ].copy()
    od_candidate_ids = set(od_retrievals["candidate_id"].astype(str))
    od_candidates = tables["candidates"][
        tables["candidates"]["candidate_id"].astype(str).isin(od_candidate_ids)
    ].copy()
    od_discovery = tables["discovery_records"][
        tables["discovery_records"]["candidate_id"].astype(str).isin(od_candidate_ids)
    ].copy()

    with pd.ExcelWriter(out,engine="openpyxl") as w:
        open_plan_df.to_excel(w,sheet_name="Open_Discovery_Plan",index=False)
        od_searches.to_excel(w,sheet_name="Searches",index=False)
        od_retrievals.to_excel(w,sheet_name="Retrievals",index=False)
        od_candidates.to_excel(w,sheet_name="Candidates",index=False)
        od_discovery.to_excel(w,sheet_name="Discovery_Metadata",index=False)
        discovery_text_df[
            discovery_text_df["candidate_id"].astype(str).isin(od_candidate_ids)
        ].to_excel(w,sheet_name="Candidate_Text",index=False)
        emergent_terms_df.to_excel(w,sheet_name="Emergent_Vocabulary",index=False)
        subject_topics_df.to_excel(w,sheet_name="Database_Topics",index=False)
        bridge_candidates_df.to_excel(w,sheet_name="Bridging_Papers",index=False)
        query_diagnostics_df.to_excel(w,sheet_name="Query_Diagnostics",index=False)
        open_discovery_integrity_df.to_excel(w,sheet_name="Integrity_Warnings",index=False)

    print("EXPORT COMPLETE:",out)
    print("Unique open-discovery candidates:",len(od_candidate_ids))
    print("Upload this workbook for the Step 1A literature-map audit.")
    return out

open_discovery_report_path = export_open_discovery_report()

# After `Run all`

The final cell should print the Google Drive path to:

`Step1A_Open_Discovery_Report.xlsx`

Upload that workbook to the chat.

Do **not** start screening all candidates or running the 121 proposition-directed searches yet. First inspect whether Step 1A discovered:
- new terminology;
- new scientific traditions;
- new experimental paradigms;
- new control architectures;
- new rival explanations;
- or evidence that the existing map is already saturated.

That inspection determines whether the main Step-1 query matrix needs another expansion.